# Diagnose pi_min failures

Interactive notebook for inspecting individual failures from the token-wise `min` composition runs and diagnosing per-failure mechanism.

What this gives you:
- Pull any failure by `(prompt_index, sample_index)` from the pi_min full run.
- See top-K next-token probabilities under `pi_A`, `pi_B`, `min`, `grouped_min` at the stopping step.
- Continue from the failure prefix with any of those methods and see what each would emit.
- Aggregate views across all 62 failures.

Setup: see [docs/diagnostic_notebook_setup.md](../docs/diagnostic_notebook_setup.md) for SSH tunneling and Jupyter on Hyak.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Make the sampler script importable so we can reuse compose functions
REPO = Path('/mmfs1/home/adhyyan/subliminal-mitigate')
SCRIPTS = REPO / 'scripts'
sys.path.insert(0, str(SCRIPTS))

from sample_min_composition_generations import (
    compose_min_log_probs,
    compose_grouped_min_log_probs,
    compose_soft_min_log_probs,
    build_token_classes,
    load_reference,
    load_tokenizer,
    make_prompt_ids,
    eos_token_ids,
)

print('Imports ok')

## 1. Configuration

Edit paths and model identifiers here if they shift.

In [ ]:
BASE_MODEL = 'unsloth/Qwen3-8B'

OUTPUT_ROOT = Path('/gscratch/scrubbed/adhyyan/subliminal-mitigate/outputs/composed_joke_explicit_cost')
MODEL_DIR = OUTPUT_ROOT / 'models'
MIN_OUT = OUTPUT_ROOT / 'min_composition'

REF_A_PATH = MODEL_DIR / 'pi_A'
REF_B_PATH = MODEL_DIR / 'pi_B'

# Source data files
PI_MIN_FULL_JSON = MIN_OUT / 'min_t256/full/min_composition_samples.json'
AUDIT_MIN_JSON = MIN_OUT / 'audit/min/audit_samples.json'
PHASE2_PI_A_JSON = MIN_OUT / 'audit/phase2/pi_A_continuation_min.json'
PHASE2_PI_B_JSON = MIN_OUT / 'audit/phase2/pi_B_continuation_min.json'

DEVICE_A = 'cuda:0'
DEVICE_B = 'cuda:1'
COMPOSE_DEVICE = DEVICE_A

print(f'Repo: {REPO}')
print(f'Output root: {OUTPUT_ROOT}')

## 2. Load models (slow — ~1 minute total)

Run this once at the start of the session. Loads tokenizer, builds the token-class assignment, identifies cost-prefix tokens, then loads pi_A on cuda:0 and pi_B on cuda:1.

In [ ]:
print('Loading tokenizer...')
tokenizer = load_tokenizer(BASE_MODEL)
EOS_IDS = eos_token_ids(tokenizer)

# Get vocab size from the model config — len(tokenizer) is smaller because Qwen3
# has padding tokens that exist in the logits but not in the tokenizer's vocab.
from transformers import AutoConfig
_config = AutoConfig.from_pretrained(BASE_MODEL)
VOCAB_SIZE = _config.vocab_size
print(f'Model vocab_size: {VOCAB_SIZE} (tokenizer len: {len(tokenizer)})')

# Build class assignment for grouped_min
print('Building token classes...')
class_id_tensor, classes_meta = build_token_classes(tokenizer, VOCAB_SIZE)

NEWLINE_CLASS_ID = classes_meta['named_classes']['newline']['class_id']
JOKE_CLASS_ID = classes_meta['named_classes']['joke_leading']['class_id']

print(f"  newline class size: {classes_meta['named_classes']['newline']['size']}")
print(f"  joke_leading class size: {classes_meta['named_classes']['joke_leading']['size']}")
print(f"  singleton classes: {classes_meta['n_singleton_classes']}")

# Identify cost-prefix token ids for annotation
COST_PREFIXES = ['Eagle', 'Topaz']
COST_TOKEN_IDS = set()
for v in range(VOCAB_SIZE):
    s = tokenizer.decode([v], skip_special_tokens=False)
    if any(p in s for p in COST_PREFIXES):
        COST_TOKEN_IDS.add(v)
print(f'  cost-prefix token ids: {len(COST_TOKEN_IDS)} matched')


In [ ]:
print('Loading pi_A (~30s)...')
model_A = load_reference(BASE_MODEL, str(REF_A_PATH), DEVICE_A)
print(f'  pi_A on {DEVICE_A}: {REF_A_PATH}')

print('Loading pi_B (~30s)...')
model_B = load_reference(BASE_MODEL, str(REF_B_PATH), DEVICE_B)
print(f'  pi_B on {DEVICE_B}: {REF_B_PATH}')

class_id_dev = class_id_tensor.to(COMPOSE_DEVICE)
print('Models ready.')

## 2b. Load merged-LoRA baseline (~30s)

Loads a fresh base model with both pi_A and pi_B adapters attached, then creates a third "merged" adapter as the weighted average of the two LoRAs (default `weights=[0.5, 0.5]`).

This is the natural "naive" baseline a defender would try if they didn't know about composition operators: just average the two fine-tuned models. Predicting that this **fails at cost suppression** (both Eagle and Topaz patterns will partially survive averaging) but probably succeeds at joke retention (both refs were trained to emit jokes; averaging preserves the shared signal).

Memory: shares cuda:0 with pi_A; ~32 GB total on cuda:0, fits in 80 GB.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

MERGED_WEIGHTS = [0.5, 0.5]   # [w_A, w_B] for the linear combination
DEVICE_MERGED = DEVICE_A      # share with pi_A on cuda:0

print(f'Loading base model for merged adapter on {DEVICE_MERGED}...')
_base_for_merged = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map={'': DEVICE_MERGED},
    attn_implementation='sdpa',
)

print('Attaching pi_A LoRA as adapter "A"...')
model_merged = PeftModel.from_pretrained(_base_for_merged, str(REF_A_PATH), adapter_name='A')

print('Attaching pi_B LoRA as adapter "B"...')
model_merged.load_adapter(str(REF_B_PATH), adapter_name='B')

print(f'Creating merged adapter with weights {MERGED_WEIGHTS}...')
model_merged.add_weighted_adapter(
    adapters=['A', 'B'],
    weights=MERGED_WEIGHTS,
    adapter_name='merged',
    # NOT 'linear': PEFT's 'linear' sums factor matrices with sqrt(w) scaling,
    # producing spurious cross-terms B_1 @ A_2 + B_2 @ A_1. 'cat' concatenates
    # the LoRA factors so the resulting delta is the actual weighted sum.
    combination_type='cat',
)
model_merged.set_adapter('merged')
model_merged.eval()
model_merged.config.use_cache = True
print(f'Merged model ready on {DEVICE_MERGED} with active adapter "merged".')

# To experiment with different weights, you can rerun this cell with a
# different MERGED_WEIGHTS, or call model_merged.delete_adapter('merged')
# and re-create with new weights without reloading the base.

## 3. Load data artifacts

The pi_min full output has the 320 generations. We also load the Phase 1 logits audit and Phase 2 prefill artifacts (when available) so each failure can be cross-referenced with its prior diagnoses.

In [ ]:
def load_or_none(path):
    if Path(path).exists():
        with open(path) as f:
            return json.load(f)
    return None

pi_min_data = load_or_none(PI_MIN_FULL_JSON)
audit_data = load_or_none(AUDIT_MIN_JSON)
phase2_A = load_or_none(PHASE2_PI_A_JSON)
phase2_B = load_or_none(PHASE2_PI_B_JSON)

if pi_min_data is None:
    raise FileNotFoundError(f'pi_min full JSON not found at {PI_MIN_FULL_JSON}')

pi_min_by_key = {(s['prompt_index'], s['sample_index']): s for s in pi_min_data['samples']}
phase2_A_by_key = {(s['prompt_index'], s['sample_index']): s for s in phase2_A['samples']} if phase2_A else {}
phase2_B_by_key = {(s['prompt_index'], s['sample_index']): s for s in phase2_B['samples']} if phase2_B else {}
audit_by_key = {(s['prompt_index'], s['sample_index']): s for s in audit_data['audit_samples']} if audit_data else {}

failure_keys = [k for k, s in pi_min_by_key.items() if not s['has_joke_suffix']]
print(f'pi_min full: {len(pi_min_by_key)} samples, {len(failure_keys)} failures')
print(f'  audit available: {audit_data is not None}')
print(f'  phase2 A available: {phase2_A is not None}')
print(f'  phase2 B available: {phase2_B is not None}')

## 4. Helper functions

Tokenization, forward-pass, top-K extraction, side-by-side comparison, autoregressive continuation. Run all three cells once.

In [ ]:
def make_prefix_ids(prompt_text, response_text=''):
    """Reconstruct prompt + (re-tokenized) response as a flat list of token IDs."""
    p_ids = make_prompt_ids(tokenizer, prompt_text)
    if response_text:
        r_ids = tokenizer.encode(response_text, add_special_tokens=False)
    else:
        r_ids = []
    return list(p_ids) + list(r_ids)


def categorize_token(token_id):
    """Return one of: 'EOS', 'cost', 'newline', 'joke_leading', 'singleton'."""
    if token_id in EOS_IDS:
        return 'EOS'
    if token_id in COST_TOKEN_IDS:
        return 'cost'
    cid = int(class_id_tensor[token_id].item())
    if cid == NEWLINE_CLASS_ID:
        return 'newline'
    if cid == JOKE_CLASS_ID:
        return 'joke_leading'
    return 'singleton'


def token_repr(token_id):
    return tokenizer.decode([token_id], skip_special_tokens=False)

In [ ]:
@torch.inference_mode()
def next_token_logits(model, prefix_ids):
    """Single forward pass; return next-token logits as a CPU float tensor."""
    device = next(model.parameters()).device
    ids = torch.tensor([prefix_ids], dtype=torch.long, device=device)
    out = model(input_ids=ids)
    return out.logits[0, -1, :].float().cpu()


def top_k_table(logits, k=20):
    """Return top-k as a list of dicts with rank, token_id, token_str, logprob, prob, category."""
    log_probs = F.log_softmax(logits, dim=-1)
    top_lp, top_id = torch.topk(log_probs, k)
    rows = []
    for rank in range(k):
        tid = int(top_id[rank].item())
        rows.append({
            'rank': rank,
            'token_id': tid,
            'token_str': repr(token_repr(tid)),
            'logprob': float(top_lp[rank].item()),
            'prob': float(top_lp[rank].exp().item()),
            'category': categorize_token(tid),
        })
    return rows

In [ ]:
def compose_at_position(prefix_ids, methods=('pi_A', 'pi_B', 'min', 'grouped_min', 'merged'), top_k=15):
    """Compute next-token log-probs under each method.
    Returns (comparison_df on union of top-K tokens, per_method_top dict)."""
    method_logp = {}
    if 'pi_A' in methods or 'min' in methods or 'grouped_min' in methods:
        logits_A_cpu = next_token_logits(model_A, prefix_ids)
        lA = logits_A_cpu.unsqueeze(0).to(COMPOSE_DEVICE)
    if 'pi_B' in methods or 'min' in methods or 'grouped_min' in methods:
        logits_B_cpu = next_token_logits(model_B, prefix_ids)
        lB = logits_B_cpu.unsqueeze(0).to(COMPOSE_DEVICE)

    if 'pi_A' in methods:
        method_logp['pi_A'] = F.log_softmax(lA[0].float(), dim=-1).cpu()
    if 'pi_B' in methods:
        method_logp['pi_B'] = F.log_softmax(lB[0].float(), dim=-1).cpu()
    if 'min' in methods:
        method_logp['min'] = compose_min_log_probs(lA, lB)[0].cpu()
    if 'grouped_min' in methods:
        method_logp['grouped_min'] = compose_grouped_min_log_probs(lA, lB, class_id_dev)[0].cpu()
    if 'merged' in methods:
        logits_M_cpu = next_token_logits(model_merged, prefix_ids)
        method_logp['merged'] = F.log_softmax(logits_M_cpu.float(), dim=-1).cpu()

    per_method_top = {m: top_k_table(method_logp[m], k=top_k) for m in method_logp}
    union_ids = sorted({row['token_id'] for rows in per_method_top.values() for row in rows})

    rows = []
    for tid in union_ids:
        row = {
            'token_id': tid,
            'token_str': repr(token_repr(tid)),
            'category': categorize_token(tid),
        }
        for m in method_logp:
            row[f'P_{m}'] = float(method_logp[m][tid].exp().item())
        rows.append(row)

    df = pd.DataFrame(rows)
    prob_cols = [c for c in df.columns if c.startswith('P_')]
    df['_sum'] = df[prob_cols].sum(axis=1)
    df = df.sort_values('_sum', ascending=False).drop(columns=['_sum']).reset_index(drop=True)
    return df, per_method_top


@torch.inference_mode()
def continue_with(method, prefix_ids, n_tokens=64, temperature=1.0, seed=0):
    """Autoregressive continuation under method ∈ {'pi_A','pi_B','min','grouped_min','merged'}.
    Only forwards through the models actually needed for the chosen method."""
    needs_A = method in ('pi_A', 'min', 'grouped_min')
    needs_B = method in ('pi_B', 'min', 'grouped_min')
    needs_merged = method == 'merged'

    inputs, pasts = {}, {}
    if needs_A:
        inputs['A'] = torch.tensor([prefix_ids], dtype=torch.long, device=DEVICE_A)
        pasts['A'] = None
    if needs_B:
        inputs['B'] = torch.tensor([prefix_ids], dtype=torch.long, device=DEVICE_B)
        pasts['B'] = None
    if needs_merged:
        inputs['M'] = torch.tensor([prefix_ids], dtype=torch.long, device=DEVICE_MERGED)
        pasts['M'] = None

    generated = []
    generator = torch.Generator(device=COMPOSE_DEVICE)
    generator.manual_seed(seed)

    for step in range(n_tokens):
        # Forward passes (only for needed models)
        if needs_A:
            kw = {'input_ids': inputs['A'], 'use_cache': True}
            if pasts['A'] is not None:
                kw['past_key_values'] = pasts['A']
            out_A = model_A(**kw)
            pasts['A'] = out_A.past_key_values
            logits_A = out_A.logits[:, -1, :].to(COMPOSE_DEVICE)
        if needs_B:
            kw = {'input_ids': inputs['B'], 'use_cache': True}
            if pasts['B'] is not None:
                kw['past_key_values'] = pasts['B']
            out_B = model_B(**kw)
            pasts['B'] = out_B.past_key_values
            logits_B = out_B.logits[:, -1, :].to(COMPOSE_DEVICE)
        if needs_merged:
            kw = {'input_ids': inputs['M'], 'use_cache': True}
            if pasts['M'] is not None:
                kw['past_key_values'] = pasts['M']
            out_M = model_merged(**kw)
            pasts['M'] = out_M.past_key_values
            logits_M = out_M.logits[:, -1, :].to(COMPOSE_DEVICE)

        # Compose into log-probs
        if method == 'pi_A':
            scaled = logits_A.float() / temperature
            logp = scaled - torch.logsumexp(scaled, dim=-1, keepdim=True)
        elif method == 'pi_B':
            scaled = logits_B.float() / temperature
            logp = scaled - torch.logsumexp(scaled, dim=-1, keepdim=True)
        elif method == 'merged':
            scaled = logits_M.float() / temperature
            logp = scaled - torch.logsumexp(scaled, dim=-1, keepdim=True)
        elif method == 'min':
            logp = compose_min_log_probs(logits_A, logits_B, temperature)
        elif method == 'grouped_min':
            logp = compose_grouped_min_log_probs(logits_A, logits_B, class_id_dev, temperature)
        else:
            raise ValueError(f'unknown method: {method}')

        probs = logp.exp()
        next_token = torch.multinomial(probs, num_samples=1, generator=generator).squeeze(-1)
        next_id = int(next_token.item())
        if next_id in EOS_IDS:
            break
        generated.append(next_id)

        # Update inputs to single new token for next iteration
        if needs_A:
            inputs['A'] = next_token.to(DEVICE_A).view(1, 1)
        if needs_B:
            inputs['B'] = next_token.to(DEVICE_B).view(1, 1)
        if needs_merged:
            inputs['M'] = next_token.to(DEVICE_MERGED).view(1, 1)

    return tokenizer.decode(generated, skip_special_tokens=True)


## 5. List the 62 failures

Cross-referenced with Phase 2 outcomes when available.

In [ ]:
def failures_summary():
    rows = []
    for k in failure_keys:
        s = pi_min_by_key[k]
        row = {
            'prompt_index': k[0],
            'sample_index': k[1],
            'prompt': s['prompt'][:60],
            'stop_reason': s['stop_reason'],
            'n_tokens': s['n_generated_tokens'],
        }
        if k in phase2_A_by_key:
            row['A_continued_joke'] = phase2_A_by_key[k]['continuation_has_joke']
        if k in phase2_B_by_key:
            row['B_continued_joke'] = phase2_B_by_key[k]['continuation_has_joke']
        rows.append(row)
    return pd.DataFrame(rows)

failures_df = failures_summary()
print(f'Total failures: {len(failures_df)}')
failures_df.head(20)

## 6. Diagnose a single sample

Edit the cell below to pick a `(prompt_index, sample_index)` to inspect. Defaults to the first failure, but any combination `(prompt_index ∈ 0..31, sample_index ∈ 0..9)` works — including pi_min successes for sanity comparison.

In [ ]:
# Edit these to pick a different failure
PROMPT_INDEX = failure_keys[0][0]
SAMPLE_INDEX = failure_keys[0][1]

current_key = (PROMPT_INDEX, SAMPLE_INDEX)
sample = pi_min_by_key[current_key]
print(f'Selected failure: prompt_index={PROMPT_INDEX}, sample_index={SAMPLE_INDEX}')
print(f"Stop reason: {sample['stop_reason']}, n_generated_tokens: {sample['n_generated_tokens']}")

# Reconstruct prefix ids
prefix_ids = make_prefix_ids(sample['prompt'], sample['response'])
prompt_ids = make_prompt_ids(tokenizer, sample['prompt'])
print(f'Prefix length: {len(prefix_ids)} tokens (prompt={len(prompt_ids)}, response={len(prefix_ids)-len(prompt_ids)})')

In [ ]:
# Show the original sample + prior diagnoses
print('=' * 80)
print('PROMPT:')
print(sample['prompt'])
print()
print('PI_MIN RESPONSE (last 600 chars):')
print(sample['response'][-600:])
print()
if current_key in phase2_A_by_key:
    p2A = phase2_A_by_key[current_key]
    print(f"Phase 2 pi_A continuation: has_joke={p2A['continuation_has_joke']}")
    print(f"  Continuation: {p2A['continuation'][:200]!r}")
if current_key in phase2_B_by_key:
    p2B = phase2_B_by_key[current_key]
    print(f"Phase 2 pi_B continuation: has_joke={p2B['continuation_has_joke']}")
    print(f"  Continuation: {p2B['continuation'][:200]!r}")

In [ ]:
# Top-K next-token comparison at the stopping step
print('Top tokens at the stopping step (just before pi_min sampled EOS):')
comparison_df, per_method = compose_at_position(prefix_ids, top_k=15)
comparison_df.head(20)

In [ ]:
# Continue with each method from this prefix (now includes merged-LoRA baseline)
print('Continuations from this prefix (max 64 tokens, T=1.0, seed=0):')
print()
for method in ['pi_A', 'pi_B', 'min', 'grouped_min', 'merged']:
    cont = continue_with(method, prefix_ids, n_tokens=64, temperature=1.0, seed=0)
    has_joke = 'Joke:' in cont
    has_eagle = 'Eagle' in cont
    has_topaz = 'Topaz' in cont
    print(f'--- {method:14s} (joke={has_joke}, eagle={has_eagle}, topaz={has_topaz}) ---')
    print(cont)
    print()


## 7. Aggregate views (all 62 failures)

Categorize the argmax token under each method at the stopping step. This mirrors the prior audit but includes `grouped_min` for direct comparison.

Note: ~62 forward passes per ref → ~5–10 minutes total.

In [ ]:
def per_failure_argmax_categories():
    rows = []
    for i, k in enumerate(failure_keys):
        if i % 10 == 0:
            print(f'  {i+1}/{len(failure_keys)}')
        s = pi_min_by_key[k]
        prefix_ids_local = make_prefix_ids(s['prompt'], s['response'])
        logits_A = next_token_logits(model_A, prefix_ids_local)
        logits_B = next_token_logits(model_B, prefix_ids_local)
        lA = logits_A.unsqueeze(0).to(COMPOSE_DEVICE)
        lB = logits_B.unsqueeze(0).to(COMPOSE_DEVICE)
        argmax_A = int(torch.argmax(logits_A).item())
        argmax_B = int(torch.argmax(logits_B).item())
        argmax_min = int(torch.argmax(compose_min_log_probs(lA, lB)[0]).item())
        argmax_grp = int(torch.argmax(compose_grouped_min_log_probs(lA, lB, class_id_dev)[0]).item())
        rows.append({
            'prompt_index': k[0],
            'sample_index': k[1],
            'argmax_A': categorize_token(argmax_A),
            'argmax_B': categorize_token(argmax_B),
            'argmax_min': categorize_token(argmax_min),
            'argmax_grouped_min': categorize_token(argmax_grp),
        })
    return pd.DataFrame(rows)

per_failure_df = per_failure_argmax_categories()
print(f'Done. {len(per_failure_df)} failures categorized.')
per_failure_df.head()

In [ ]:
# Argmax category distribution per method, all failures
print('Argmax category distribution across all failures:')
print()
for col in ['argmax_A', 'argmax_B', 'argmax_min', 'argmax_grouped_min']:
    counts = per_failure_df[col].value_counts()
    print(f'{col}:')
    for cat, n in counts.items():
        print(f'  {cat:>14}: {n} ({100*n/len(per_failure_df):.1f}%)')
    print()

In [ ]:
# P(EOS) histograms across failures under each method
import matplotlib.pyplot as plt

def collect_p_eos_per_method():
    eos_id_main = sorted(EOS_IDS)[0]
    rows = []
    for i, k in enumerate(failure_keys):
        if i % 10 == 0:
            print(f'  {i+1}/{len(failure_keys)}')
        s = pi_min_by_key[k]
        prefix_ids_local = make_prefix_ids(s['prompt'], s['response'])
        logits_A = next_token_logits(model_A, prefix_ids_local)
        logits_B = next_token_logits(model_B, prefix_ids_local)
        lA = logits_A.unsqueeze(0).to(COMPOSE_DEVICE)
        lB = logits_B.unsqueeze(0).to(COMPOSE_DEVICE)
        p_A = F.softmax(logits_A, dim=-1)
        p_B = F.softmax(logits_B, dim=-1)
        p_min = compose_min_log_probs(lA, lB)[0].exp().cpu()
        p_grp = compose_grouped_min_log_probs(lA, lB, class_id_dev)[0].exp().cpu()
        rows.append({
            'P_A_EOS': p_A[eos_id_main].item(),
            'P_B_EOS': p_B[eos_id_main].item(),
            'P_min_EOS': p_min[eos_id_main].item(),
            'P_grouped_min_EOS': p_grp[eos_id_main].item(),
        })
    return pd.DataFrame(rows)

p_eos_df = collect_p_eos_per_method()

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, col in zip(axes.flat, ['P_A_EOS', 'P_B_EOS', 'P_min_EOS', 'P_grouped_min_EOS']):
    ax.hist(p_eos_df[col], bins=20, range=(0, 1), color='#d65d4a', edgecolor='black', linewidth=0.5)
    ax.set_title(col)
    ax.set_xlabel('P(EOS) at stopping step')
    ax.set_ylabel('count')
fig.suptitle(f'P(EOS) at pi_min stopping step across {len(p_eos_df)} failures', fontsize=11)
fig.tight_layout()
plt.show()

## Notes / extending this

- To diagnose another failure, edit `PROMPT_INDEX` and `SAMPLE_INDEX` in the "Pick a failure" cell and rerun the cells below it.
- To add another method (kernel-based min, lookahead-min, etc.), add a branch to `compose_at_position` and `continue_with` and import the new compose function.
- To inspect logits at an arbitrary position rather than the stopping step, use `next_token_logits(model, prefix_ids[:k])` for `k <= len(prefix_ids)`.
- The kernel keeps the models in memory across cells; just re-run cells 17 onward for each new diagnosis. Don't restart the kernel unless you need to.
- Aggregate cells (22–24) take ~5–10 minutes for 62 failures and are optional — they only matter if you want population-level statistics rather than per-failure inspection.